In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multi-GPU in Practice

In that section we built data parallelism by hand, and a second
GPU made the training steps of LeNet slower: each GPU had too little work
to outweigh even a small exchange of gradients. A larger model gives each
GPU more work for every byte it exchanges, so it should fare better. How
much better, once the exchange is organized well? And what can several
GPUs do for a model that is too large for any one of them?

A ResNet-18 on Fashion-MNIST images enlarged to 64×64 pixels gives each
GPU far more work than LeNet. It performs 2.2 billion multiply-adds per image, about
fifty for every byte of its 45 MB of gradients, so the work that a second
GPU takes over is large compared with the gradients that the two GPUs must
exchange. Whether this advantage survives depends on how the exchange is
organized. The hand-written loop of that section waited for
the whole backward pass before it communicated, although the gradients of
the last layers were ready long before those of the first.

It also issued the work of every GPU from one Python thread and routed all
gradients through device 0.

Data parallelism also leaves one cost untouched: every GPU keeps a full
copy of the training state. The GPT of that section with
36 blocks instead of 18 has 1.9 billion parameters. Its weights, their
gradients, and Adam's moment estimates take 29 GiB, more than the 24 GiB
of an RTX 4090. Data parallelism would place these 29 GiB on every one of
our four GPUs, so four GPUs cannot train this model any more than one can.

We first replace the hand-written loop with each framework's
production tool for data parallelism, which starts communicating while the
backward pass is still running. We measure how ResNet-18 then scales on two
and four GPUs, and compare the measurement with the cost model of
that section. We then divide the training state among the GPUs
instead of copying it, which reduces the memory per GPU but adds
communication.
that section applies both to a transformer.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
import json
import os
import random
import sys

## Production Data Parallelism

Production data parallelism computes what the loop of
that section computes. Every GPU holds a copy of the model,
computes the gradient of the loss on its share of each minibatch, and
applies the average of the gradients of all shares. What changes is how the
work is organized, and above all when the communication starts.

PyTorch changes two more things. Every GPU gets a process of its own,
and NCCL replaces our star with the ring of that section,
or with a tree for small messages, so that no device carries the traffic of
all the others.

### Overlapping Communication with the Backward Pass

The backward pass computes gradients from the last layer to the first, so
the gradients of the last layers are complete long before those of the
first. Our loop in that section nevertheless waited for the
whole backward pass before it started the allreduce, and the links between
the GPUs sat idle in the meantime. The production tools group the
gradients into *buckets* of a few tens of megabytes, in the order in which
the backward pass produces them, and start the allreduce of a bucket as
soon as its last gradient exists [@Li.Zhao.Varma.ea.2020]. While one
bucket travels between the GPUs, the GPUs compute the gradients of the
next (the figure).

![Top: the allreduce waits for the whole backward pass. Bottom: the
gradients are grouped into buckets, and each bucket's allreduce starts as
soon as the backward pass has produced it. Only the last bucket runs after
the backward pass has ended.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-ddp-overlap.svg)

The size of a bucket is a compromise between the two parts of an
allreduce's time, a fixed cost $\alpha$ per call and the bytes divided by
the bandwidth $\beta$. Small buckets start early but incur $\alpha$ many
times. A single bucket for all gradients incurs $\alpha$ once, but it waits
for the whole backward pass. Whatever the size, the last bucket holds the
gradients of the first layers, which the backward pass computes last. Its
allreduce has nothing left to overlap with, so its time adds to every step.
Overlap therefore shortens the communication of a step, but it cannot
remove it.

In PyTorch, `DistributedDataParallel` (DDP) implements bucketing. It
registers a hook on every parameter that marks the parameter's gradient as
ready, and it starts the allreduce of a bucket once all of the bucket's
gradients are ready. A bucket holds up to 25 MiB (the argument
`bucket_cap_mb`), and the buckets are filled in the reverse order of the
model's parameters, which is close to the order of the backward pass.
ResNet-18's 45 MB of gradients form three buckets of about 9, 26, and 9 MB.
The last of them holds the gradients of the first two thirds of the layers,
which have few channels and therefore few weights. The allreduces run on a
CUDA stream of their own, so they proceed while the backward kernels run.

### The Execution Model

In that section one Python thread issued the kernels of every
GPU, one GPU after the other. The time that the host spends on a step then
grows with the number of GPUs, and for a model with many small kernels the
host becomes the bottleneck, as it did for LeNet. PyTorch instead starts
one process per GPU. Every process runs an ordinary training script on its
own GPU, so no GPU waits for the host to serve the others. The older
`nn.DataParallel` drives all GPUs from one process, as our loop did, and
PyTorch's documentation recommends DDP even within one machine.

The processes find each other through
`torch.distributed.init_process_group`, which connects them through NCCL,
NVIDIA's library of collective operations, and DDP sends its allreduces
through NCCL. The launcher `torchrun` starts $k$ copies of a script and
tells each copy its *rank*, its index from 0 to $k-1$, through environment
variables. A notebook is a single process, so we write each script as a
string and start it with the following helper. It collects the lines of
JSON that the processes print, process by process in the order of their
ranks. Every script begins with `init_distributed`: it pins its process to
the GPU that `torchrun` assigned to it and joins the process group.

In [ ]:

import glob
import subprocess
import tempfile

def torchrun(script, nproc, *args, env=None):
    """Run Python source in nproc processes; return their JSON lines."""
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, 'script.py')
        with open(path, 'w') as f:
            f.write(script)
        launcher = os.path.join(os.path.dirname(sys.executable), 'torchrun')
        # Each process prints to a file of its own (--redirects 1), so that
        # the lines of different processes cannot interleave
        run = subprocess.run(
            [launcher, '--standalone', f'--nproc-per-node={nproc}',
             '--log-dir', tmp, '--redirects', '1', path, *map(str, args)],
            capture_output=True, text=True, env={**os.environ, **(env or {})})
        if run.returncode != 0:
            raise RuntimeError(run.stderr[-3000:])
        logs = sorted(glob.glob(f'{tmp}/*/attempt_0/*/stdout.log'),
                      key=lambda log: int(log.split(os.sep)[-2]))  # By rank
        return [json.loads(line) for log in logs for line in open(log)
                if line.startswith('{')]

def init_distributed():
    """Join the process group of torchrun; return the rank, the number of
    processes, and the GPU of this process."""
    device = torch.device('cuda', int(os.environ['LOCAL_RANK']))
    torch.cuda.set_device(device)
    torch.distributed.init_process_group('nccl', device_id=device)
    return (torch.distributed.get_rank(), torch.distributed.get_world_size(),
            device)

### The Training Step

Our model is the ResNet-18 of that section with a stem for small
images: a 3×3 convolution without max-pooling, followed by four stages of
two residual blocks each, with He initialization as in
that section. We save it, since later chapters train it on several
GPUs.

In [ ]:
def resnet18(num_classes, in_channels=1):
    """ResNet-18 for small images: a 3x3 stem without max-pooling, He
    initialization."""
    def stage(num_channels, first=False):
        return [d2l.Residual(num_channels, use_1x1conv=not first,
                             strides=1 if first else 2),
                d2l.Residual(num_channels)]
    net = nn.Sequential(
        # No stem bias: the batch normalization after the conv removes it
        nn.Conv2d(in_channels, 64, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(64), nn.ReLU(),
        *stage(64, first=True), *stage(128), *stage(256), *stage(512),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, num_classes))
    with torch.no_grad():  # Create the lazy parameters of the blocks
        net.eval()(torch.zeros(1, in_channels, 32, 32))
    for m in net.modules():  # He initialization, zero biases
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return net.train()

As in that section, every GPU keeps its own copy of the
training set, here enlarged once to 64×64 pixels, which takes 983 MB in
fp32. The input pipeline then stays out of our measurements, and the only
data that cross between GPUs are gradients. We also count the gradient
bytes $N$ that every step must sum across the GPUs:

In [ ]:
d2l.FashionMNIST()                   # Download once, before any process runs
num_params = d2l.num_params(resnet18(10))
N = 4 * num_params                   # Gradient bytes in fp32
print(f'{num_params / 1e6:.1f}M parameters, N = {N / 1e6:.0f} MB')

The training script follows. Two parts make it data-parallel:
`init_distributed`, which joins the process to the others, and the `DDP`
wrapper around the model. The rest is a single-GPU training loop. Every
process takes its own slice of each global minibatch of $kb$ examples.
All processes draw the same random permutation, because they share a seed,
and the process of rank $r$ takes the $r$-th of the $k$ slices of $b$
examples. For data that stay in host memory, `DistributedSampler` performs
the same split for a `DataLoader`. The script times its steps with
`d2l.Benchmark` (that section) and prints the seconds per step.

In [ ]:
TRAIN_DDP = r'''
import json, sys, torch
import torch.distributed as dist
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from d2l import torch as d2l

rank, k, device = d2l.init_distributed()
torch.backends.cudnn.benchmark = True  # Tune convolutions for fixed shapes
b, num_steps, seed = map(int, sys.argv[1:])  # b examples per GPU and step
torch.manual_seed(seed)                # The same permutations everywhere

data = d2l.FashionMNIST().train        # A copy on every GPU, at 64 x 64
X = F.interpolate(data.data.to(device)[:, None] / 255, 64, mode='bilinear')
y = data.targets.to(device)
model = DDP(d2l.resnet18(10).to(device), device_ids=[device])
opt = torch.optim.SGD(model.parameters(), lr=0.1)

def minibatches():
    """This process's b examples of every global minibatch of k * b."""
    while True:
        perm = torch.randperm(len(y), device=device)
        for s in range(len(y) // (k * b)):
            yield perm[(s * k + rank) * b:][:b]

def train_step(i):
    opt.zero_grad(set_to_none=True)
    F.cross_entropy(model(X[i]), y[i]).backward()
    opt.step()

batches = minibatches()
t = d2l.Benchmark(lambda: train_step(next(batches)), warmup=10,
                  repeats=num_steps).time
print(json.dumps({'rank': rank, 'step_time': t}))
dist.destroy_process_group()
'''

### A One-Step Check

Before timing anything, we check, as in that section, that a
step on two GPUs computes what we expect. We compare it with steps on one
GPU, from the same initialization and on the same 256 examples, in full
fp32 arithmetic so that rounding differences stay small. The layer to
watch is batch normalization (that section). It normalizes
every channel with the mean and the variance over the minibatch, and when
the minibatch is split across GPUs, these statistics can cover either one
GPU's shard or the whole minibatch.

The script below takes one DDP step on $k$ GPUs, each GPU on its half of
256 random images. Process 0 also keeps an unwrapped copy of the model and
takes two reference steps on its GPU: one on all 256 examples at once, and
one that accumulates the gradients of the same two halves before it
updates the parameters.

In [ ]:
CHECK_DDP = r'''
import copy, json, sys, torch
import torch.distributed as dist
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from d2l import torch as d2l

rank, k, device = d2l.init_distributed()
torch.backends.cudnn.allow_tf32 = False  # Full fp32 convolutions
torch.manual_seed(int(sys.argv[1]))      # The same model and data everywhere
X = torch.rand(256, 1, 64, 64, device=device)
y = torch.randint(0, 10, (256,), device=device)
model = d2l.resnet18(10).to(device)
ref = copy.deepcopy(model)               # An unwrapped copy for one GPU

def update(net, shards):
    """Parameter change of one SGD step on the mean loss over the shards."""
    before = [p.detach().clone() for p in net.parameters()]
    for X_j, y_j in shards:
        (F.cross_entropy(net(X_j), y_j) / len(shards)).backward()
    with torch.no_grad():
        for p, b in zip(net.parameters(), before):
            p -= 0.1 * p.grad
    return [p.detach() - b for p, b in zip(net.parameters(), before)]

ddp = update(DDP(model, device_ids=[device]),
             [(X.chunk(k)[rank], y.chunk(k)[rank])])
if rank == 0:
    gap = lambda u: max((a - b).abs().max().item() for a, b in zip(ddp, u))
    halves = update(copy.deepcopy(ref), list(zip(X.chunk(k), y.chunk(k))))
    print(json.dumps({'largest': max(u.abs().max().item() for u in ddp),
                      'full': gap(update(ref, [(X, y)])),
                      'halves': gap(halves)}))
dist.destroy_process_group()
'''
if d2l.num_gpus() >= 2:
    r = d2l.torchrun(CHECK_DDP, 2, random.getrandbits(32))[0]
    print(f"largest update {r['largest']:.1e}\n"
          f"largest difference from one GPU on all 256: {r['full']:.1e}\n"
          f"largest difference from one GPU on two accumulated halves: "
          f"{r['halves']:.1e}")

The DDP step agrees with the accumulated halves to rounding error, well
under one percent of the largest update, but it differs from the step on
all 256 examples by more than one percent of the largest update. DDP normalizes
each GPU's shard with the statistics of that shard, so it computes what
one GPU computes when it accumulates the gradients of the same shards
(that section). The mean and the variance over 128
examples are already close to those over the whole minibatch. Over a few
examples per GPU they become noisy, and `nn.SyncBatchNorm` then computes
them over the whole minibatch, which adds collectives to every batch
normalization layer (exercise 7).

## Scaling on Four GPUs

We measure both conventions of that section: weak
scaling, which holds the minibatch per GPU fixed while the number of GPUs
$k$ grows and reports examples per second, and strong scaling, which holds
the global minibatch fixed and reports how much sooner a step finishes. The
function `step_time(k, b)` returns the seconds per step on $k$
GPUs with $b$ examples per GPU, averaged over 50 steps after a warm-up.

In [ ]:
def step_time(k, b, num_steps=50):
    """Seconds per DDP step on k GPUs with b examples per GPU."""
    ranks = d2l.torchrun(TRAIN_DDP, k, b, num_steps, random.getrandbits(32))
    return max(r['step_time'] for r in ranks)

Weak scaling first, with 256 examples per GPU:

In [ ]:
ks = [k for k in (1, 2, 4) if k <= d2l.num_gpus()]
weak = {k: step_time(k, 256) for k in ks}
for k in ks:
    print(f'{k} GPU(s): {k * 256 / weak[k]:5.0f} examples/s, speedup '
          f'{k * weak[1] / weak[k]:.2f}, efficiency {weak[1] / weak[k]:.0%}')

Two GPUs process about 1.7 times as many examples per second as one, and
four GPUs a little over three times as many. The *efficiency*, the speedup divided
by $k$, falls gradually as GPUs are added. Under weak
scaling every GPU does the same work at every $k$. The efficiency is
therefore the ratio of the single-GPU step time to the step time on $k$
GPUs, and what it loses is the communication that the step does not hide.

Strong scaling holds the global minibatch at 512, so each of the $k$ GPUs
processes $512/k$ examples while the allreduce keeps its size:

In [ ]:
strong = {k: step_time(k, 512 // k) for k in ks}
for k in ks:
    print(f'{k} GPU(s), {512 // k} examples each: speedup '
          f'{strong[1] / strong[k]:.2f}, efficiency '
          f'{strong[1] / strong[k] / k:.0%}')
d2l.plot(ks, [ks, [k * weak[1] / weak[k] for k in ks],
              [strong[1] / strong[k] for k in ks]], 'GPUs', 'speedup',
         legend=['linear', 'weak (256 per GPU)', 'strong (512 in all)'],
         fmts=('k:', '-o', '--s'))

At two GPUs the two conventions give about the same speedup, because 256
and 512 examples keep an RTX 4090 about equally busy. At four GPUs they
diverge. Strong scaling leaves each GPU 128 examples per step, so the
step's computation shrinks while its communication does not, and the
efficiency falls well below that of weak scaling. The same mechanism made
LeNet slower on two GPUs; here its effect is much weaker.

### Predicted and Measured Communication

The cost model the equation of that section
predicts the time of a step on $k$ GPUs from the compute time of a shard of
$B/k$ examples, the number $m$ of allreduce calls with fixed cost $\alpha$
each, and the gradient bytes $N$ at bandwidth $\beta$:

$$
t_{\textrm{step}}(k) \approx t_{\textrm{compute}}(B/k) + m\,\alpha +
\frac{2N}{\beta}.
$$

Under weak scaling the first term does not change with $k$, so the
difference $t_{\textrm{step}}(2) - t_{\textrm{step}}(1)$ is the
communication that a step exposes. For 45 MB of gradients in a few calls,
the bandwidth term dominates. The model counts the whole allreduce, so
$2N/\beta$ is the exposed time if nothing overlaps. Overlap as in
the figure lowers the exposed time: only the allreduce of
the last bucket remains exposed, provided that the earlier buckets finish
while the backward pass still runs.

As in that section, we define $\beta$ as $2N$ divided by
the time of a large allreduce on two GPUs, so $\beta$ counts the bytes
that leave and enter a GPU together. We measure it with a bare allreduce of
256 MB per GPU, the same size as there, through the library that the
training uses:

`ALLREDUCE` times `torch.distributed.all_reduce` through NCCL in every
process of a two-process launch.

In [ ]:
ALLREDUCE = r'''
import json, torch
import torch.distributed as dist
from d2l import torch as d2l

_, _, device = d2l.init_distributed()
x = torch.zeros(2**26, device=device)   # 64M floats: 256 MB
t = d2l.Benchmark(lambda: dist.all_reduce(x), repeats=10).time
print(json.dumps({'beta': 2 * x.nbytes / t}))
dist.destroy_process_group()
'''
def allreduce_beta(env=None):
    """Bandwidth of a bare 256 MB allreduce on two GPUs, in bytes/s."""
    return min(r['beta'] for r in d2l.torchrun(ALLREDUCE, 2, env=env))

In [ ]:
if d2l.num_gpus() >= 2:
    beta = allreduce_beta()
    print(f'beta = {beta / 1e9:.1f} GB/s; k=2: predicted 2N/beta = '
          f'{1e3 * 2 * N / beta:.0f} ms, measured exposed '
          f'{1e3 * (weak[2] - weak[1]):.0f} ms per step')

The measured exposed time is close to $2N/\beta$, the time of an
allreduce that nothing overlaps. Overlap hides far less of the
communication on this machine than the bucket model predicts.

DDP's last bucket holds 8.7 of the 45 MB of gradients, so with full overlap
a step would expose a fifth of $2N/\beta$, about 4 ms.

The bucket model makes three assumptions: every allreduce starts as soon
as its bucket is complete, it runs as fast as the bare benchmark, and the
backward kernels beside it run as fast as they do alone. A trace of one
step, which records when every kernel starts and ends, tests them
(exercise 4). On this machine the third fails: the
kernels of an allreduce take a share of the GPU's processors and memory
bandwidth, and the backward kernels beside them run more slowly.

In a trace of the step on two GPUs, the backward kernels take about a
tenth longer than in the same step without communication, and the
allreduces take more than twice as long as the bare benchmark. Together
they lengthen the step by about 15 ms.

On a machine whose GPUs share an NVLink domain, each GPU exchanges data
with the others at 450 GB/s in each direction on the H100 (NVLink 4) and
at 900 GB/s on the B200 (NVLink 5), hundreds of times the rate measured
here (that section). The allreduce of ResNet-18's
gradients then takes well under a millisecond, and the cost model predicts
near-linear scaling for this model with or without overlap.

### The Collective Transport

At $k = 2$ each GPU sends and receives at $\beta/2$ in each direction,
about 2.5 GB/s. A copy between two of our GPUs, staged through host memory,
can run at up to 23 GB/s (that section), about nine
times as fast. The cause lies in how NCCL moves data between GPUs that lack
peer-to-peer access. It falls back to buffers in host memory, and by
default GPU kernels copy the data into and out of these buffers with
ordinary loads and stores across PCIe. The copy engines, the units that
perform a GPU's memory copies, move the same bytes faster. An undocumented
NCCL parameter, `NCCL_SHM_USE_CUDA_MEMCPY=1`, makes NCCL use them. It
exists in the NCCL versions that our pinned frameworks bundle, and NCCL
2.31 removed it. We measure the bare allreduce with the parameter set:

In [ ]:
if d2l.num_gpus() >= 2:
    beta_ce = allreduce_beta({'NCCL_SHM_USE_CUDA_MEMCPY': '1'})
    print(f'beta = {beta / 1e9:.1f} GB/s by default, '
          f'{beta_ce / 1e9:.1f} GB/s with the copy engines '
          f'({beta_ce / beta:.1f}x)')

The bandwidth rises nearly five-fold, and with it the predicted
communication time of ResNet-18 falls from about 18 ms to about 4 ms.

PyTorch 2.11 bundles NCCL 2.28.9. On our machine, DDP training with the
parameter set deadlocks within seconds, although the same collectives
complete in the bare benchmark. Exercise 5
reproduces both outcomes under a time limit. We therefore train with
NCCL's default.

One environment variable changed $\beta$ nearly five-fold here. Whether such a
setting is safe has to be checked on the training run itself, not only on
a benchmark of bare collectives.

## Sharding the Training State

Data parallelism makes training faster, but it makes no room for a larger
model: every GPU holds all of the weights, the gradients, and the
optimizer state. With Adam in fp32 these take 16 bytes per parameter
(that section), so a 24 GiB card holds at most 1.6 billion
parameters before it stores a single activation. The 36-block GPT from the
start of this section has 1.9 billion parameters. At 16 bytes each they
take 29 GiB, which no single GPU of ours holds, and data parallelism would
need these 29 GiB on every GPU.

### Partitioning the Optimizer State, Gradients, and Weights

Each GPU needs a full copy of the weights only while it computes with them,
and it never needs the full gradient or the full optimizer state. The ZeRO
family of methods [@Rajbhandari.Rasley.Ruwase.ea.2020] removes the
redundant copies in three stages. Stage 1 partitions the optimizer state:
each GPU keeps Adam's moment estimates for $1/k$ of the parameters and
updates only those parameters. Stage 2 also partitions the gradients, and
stage 3 also partitions the weights. the table lists what each
GPU then holds.

<!-- tbl-caption:Training state per GPU with Adam in fp32 on $k$ GPUs, and for the GPT with 36 blocks and 1.9 billion parameters on $k = 4$. -->

| each GPU holds | weights | gradients | Adam state | bytes per parameter | GPT, $k = 4$ |
|---|---|---|---|---|---|
| data parallelism | all | all | all | 16 | 28.6 GiB |
| ZeRO stage 1 | all | all | $1/k$ | $8 + 8/k$ | 17.9 GiB |
| ZeRO stage 2 | all | $1/k$ | $1/k$ | $4 + 12/k$ | 12.5 GiB |
| ZeRO stage 3 (FSDP) | $1/k$ | $1/k$ | $1/k$ | $16/k$ | 7.1 GiB |

The collectives follow from the identity of that section: an
allreduce is a reduce-scatter followed by an all-gather. In stages 1 and
2 the reduce-scatter leaves each GPU with the summed gradient of the
parameters it owns, which is all that its optimizer needs. Each GPU then
updates its $1/k$ of the parameters, and an all-gather assembles the
updated parameters on every GPU. The two phases move the same bytes as the
allreduce of data parallelism, now separated by the update. Stages 1 and 2
therefore reduce the memory per GPU at no cost in communication. For our
GPT, stage 1 on four GPUs already brings the training state below the
24 GiB of one card.

Stage 3 is known as *fully sharded data parallelism* (FSDP)
[@Zhao.Gu.Varma.ea.2023]. Each GPU keeps only its shard of every
block's weights, and an all-gather assembles a block's weights just before
the block computes and releases them right after
(the figure). The backward pass gathers each block's
weights once more, releases them after the block's backward pass, and
reduce-scatters the block's gradients. A step thus performs two all-gathers
and one reduce-scatter of $N$ bytes, three phases where an allreduce has
two, so FSDP communicates 1.5 times as much as data parallelism. No full
copy of a block lives longer than the block's computation, so the memory
per GPU is $16/k$ bytes per parameter plus the activations and one or two
gathered blocks.

![The life of one block's weights under FSDP on one GPU. Between uses the
block is sharded, and each GPU holds $1/k$ of it; the other blocks stay
sharded throughout. An all-gather assembles the whole block for its forward
pass, after which it is released. A second all-gather assembles it for the
backward pass, after which it is released again, and a reduce-scatter
leaves each GPU with its share of the
gradients.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-fsdp-lifecycle.svg)

the table lists the four collectives that data-parallel and
sharded training use.

<!-- tbl-caption:The collective operations behind data-parallel and sharded training. -->

| collective | what every GPU ends with | where it appears |
|---|---|---|
| allreduce | the full sum | gradients in data parallelism |
| reduce-scatter | one part of the sum | gradients in ZeRO and FSDP |
| all-gather | every part, concatenated | weights in ZeRO and FSDP |
| all-to-all | a different part from each GPU | tokens routed to the experts of a mixture of experts (that section) |

In PyTorch, `fully_shard` from `torch.distributed.fsdp` applies FSDP to a
module. It is called once for every transformer block and once for the
whole model, so that every block becomes a unit that is gathered and
released on its own. The older `FullyShardedDataParallel` wrapper still
works, and PyTorch's documentation recommends `fully_shard` for new code.
Stage 1 alone is available as
`torch.distributed.optim.ZeroRedundancyOptimizer`. DDP itself needs more
than the first row of the table: its buckets hold a second
copy of the gradients, 4 more bytes per parameter, unless the argument
`gradient_as_bucket_view=True` makes the gradients views into the buckets.

### Memory and Communication

Sharding reduces memory and adds communication, and the cost model says
how long the communication takes. For our GPT, the fp32 gradients take
$N$ = 7.7 GB, four bytes per parameter. On $k$ GPUs, a ring allreduce
makes every GPU send $2(k-1)N/k$ bytes and receive as many
(that section), $1.5N$ in each direction at $k = 4$ against
$N$ at $k = 2$. If every GPU keeps the bandwidth $\beta$ of about 5 GB/s
that we measured on two, one allreduce of data parallelism takes
$3N/\beta$, about 5 s, and the three phases of FSDP take $4.5N/\beta$,
about 7 s.

The computation of a step is much shorter. The forward and the backward
pass take about six floating-point operations per parameter and token
(that section). Two sequences of 1,024 tokens per GPU, which
fit next to the 7.1 GiB of stage 3 with the activation checkpointing of
that section, take about
$6 \cdot 1.9 \cdot 10^9 \cdot 2048 \approx 2.4 \times 10^{13}$
operations. At the RTX 4090's full tf32 rate of about 80 TFLOP/s that is
0.3 s, and 0.4 s with the forward pass that checkpointing repeats. On our
machine, sharding lets the model train on four GPUs, at a few percent of
the speed that its computation would allow. Over NVLink 5, at 900 GB/s in
each direction, the same three phases take about 20 ms, and FSDP is the
standard way to train models whose state does not fit on one GPU.

The arithmetic also says when to shard. A model whose training state fits
on one GPU next to its activations trains fastest with data parallelism.
On a 24 GiB card with Adam in fp32 that limit lies at about a billion
parameters, fewer for long sequences. The 18-block GPT of
that section sits at this limit. Stage 3 would shrink
its 15.1 GiB per GPU to 3.8 GiB at $k = 4$ and free 11 GiB for
activations, but on our machine the extra communication would take far
more time than the larger micro-batches save.

Past the limit, ZeRO stages 1 and 2 reduce the memory at no cost in
communication, and stage 3 removes the remaining copies with 1.5 times
the communication. Splitting the weight matrices of every layer across
GPUs instead, *tensor parallelism* (exercise 8), and
the forms of parallelism that span several machines belong to the
Language Models part.

We can now train a model on all GPUs of one machine with the tools that
practitioners use, predict from the cost model what a step will cost, and
check the prediction against a measurement. Our measurements also show how
much depends on the connection between the GPUs: when they exchange data
through host memory, communication sets the pace long before the
arithmetic does. that section applies the method of this
chapter to a transformer, where precision, compilation, and data
parallelism interact.

## Summary

Production data parallelism computes what the hand-written loop computes,
but the framework writes the collectives, and the allreduce of each bucket
of gradients starts while the backward pass is still running. A one-step
check against a single GPU shows what the data-parallel step computes,
including how batch normalization treats a split minibatch. On our four
RTX 4090s ResNet-18 gains from every added GPU, but its exposed
communication is close to the cost model's time for an allreduce that
nothing overlaps: on this machine, overlap hides little of it. ZeRO and
FSDP divide the 16 bytes per parameter of the training state among the
GPUs, stages 1 and 2 at no cost in communication and stage 3 at 1.5 times
the communication of data parallelism.

## Exercises

1. [code] **Splitting the data.** In `TRAIN_DDP` every process takes its
   own slice of each global minibatch: all processes draw the same
   permutation, and the process of rank $r$ takes the slice that starts at
   $(sk + r)b$ in step $s$.
    1. Remove the rank from this offset, so that every process takes the
       same $b$ examples. Predict what `step_time` reports at $k = 2$
       relative to the correct script, and which global minibatch the
       averaged gradient corresponds to. Then check, with each process
       printing a checksum of its first minibatch.
    1. Restore the offset and give each process a seed of its own. Which
       property of the split is lost, and does it matter for training?
    1. Name a check that catches the first mistake and is cheap enough to
       run at every launch.

2. **Weak-scaling extrapolation.** Suppose that a step on $k$ GPUs takes
   the single-GPU step time $t_1$ plus an exposed communication time that
   is proportional to the per-GPU ring traffic of
   that section, $2(k-1)N/k$.
    1. Express the weak-scaling efficiency $E(k)$ through a single
       constant, fit the constant to your measurement at $k = 2$, and
       predict $E(4)$. Compare your prediction with the measured $E(4)$
       and with the prediction of a model in which the exposed time is the
       same for every $k \geq 2$.
    1. Predict $E(8)$ for an eight-GPU machine of the same design as ours,
       on which every transfer between GPUs is staged through host memory
       (that section). How does the total ring traffic
       through host memory grow with $k$, and in which direction should
       this move a measured $E(8)$ relative to your prediction?
    1. With gradient accumulation (that section), each
       GPU runs $a$ micro-batches of 256 examples per optimizer step and
       communicates only after the last one. Starting from your measured
       $E(4)$, find the smallest $a$ that raises the efficiency at $k = 4$
       to 95%. What happens to the global minibatch, and why does that
       matter (that section)?

3. **Cost of sharding.** ● Count communication per GPU and step in units
   of $N$, the bytes of one full set of fp32 gradients.
    1. A variant of ZeRO stage 3 keeps every gathered block from its
       forward pass until its backward pass. Which collective does this
       remove, and how much of the memory saving of stage 3 does it give
       up, in bytes per parameter of the table?
    1. Mixed-precision FSDP gathers the weights in bf16 and reduce-scatters
       the gradients in bf16, while every GPU keeps its shard of the fp32
       weights and of Adam's state. Compare its bytes per step with those
       of data parallelism in fp32, and its memory per parameter with the
       last row of the table.
    1. For the GPT with 36 blocks and 1.9 billion parameters on our four
       GPUs, and with the $\beta$ that you measured, find the number of
       tokens per GPU and step at which the computation, 6 operations
       per parameter and token at 80 TFLOP/s, takes as long as the communication of
       stage 3. How many sequences of 1,024 tokens is that, and do their
       activations fit next to the training state
       (that section)? How does gradient accumulation
       change the answer when FSDP gathers the weights anew for every
       micro-batch?

4. [code] **Bucket size.** ● Model the overlap of the figure
   as follows. The backward pass takes time $t_b$ and releases the
   gradients in $n$ buckets of equal size, bucket $i$ at time $i\,t_b/n$.
   The allreduces of the buckets run one at a time and in order, and each
   takes $\alpha + \tau/n$, where $\tau = 2N/\beta$ is the communication
   time of all gradients and $\alpha$ is a fixed cost per collective. The
   exposed communication is the time from the end of the backward pass to
   the end of the last allreduce.
    1. Show that the exposed communication is

        $$t_{\text{exposed}}(n) = \max\Bigl(\alpha + \frac{\tau}{n},\;
        n\alpha + \tau - \frac{n-1}{n}\,t_b\Bigr).$$

    1. Find the $n$ that minimizes $t_{\text{exposed}}$ when $\tau$ is well
       below $t_b$, and again when $\tau > t_b$. What does a bucket that is
       too small cost, and what does one that is too large cost?
    1. Pass `bucket_cap_mb` to `DDP` in `TRAIN_DDP`, sweep it from 1 MB to
       100 MB at $k = 2$ with `step_time`, and plot the exposed
       communication against the bucket size. Estimate $t_b$ as two thirds
       of the single-GPU step time and $\tau$ as $2N/\beta$. Is the
       measured optimum consistent with your model, and which $\alpha$ does
       it imply?
    1. At the default buckets, the exposed communication is close to
       $\tau$ rather than to the time of the last bucket. Record a few
       steps at $k = 2$ with `torch.profiler.profile`, once as they are and
       once with the backward pass inside `model.no_sync()`, which skips
       the allreduces. Compare the time from the first to the last backward
       kernel in the two runs, and find when each allreduce kernel starts
       and ends. Which assumption of the model fails on our machine?

5. [code] **Transport configuration.** ● `allreduce_beta` compares NCCL's
   default transport with the copy engines that `NCCL_SHM_USE_CUDA_MEMCPY=1`
   selects, at a single payload of 256 MB. The parameter affects only the
   transport through host memory that NCCL uses between GPUs that lack
   peer-to-peer access.
    1. Launch `ALLREDUCE` once with `NCCL_DEBUG=INFO` and find the transport
       that NCCL reports for each channel. Predict whether the parameter can
       change anything on your machine.
    1. Extend the comparison to payloads from 1 MB to 256 MB and plot the
       bandwidth of both transports against the payload. Where does each
       saturate, and does the five-fold gap hold at every size?
    1. Launch `TRAIN_DDP` on two GPUs with the parameter set (the `env`
       argument of `d2l.torchrun`), once with the default buckets and once
       with `bucket_cap_mb` large enough that all gradients form a single
       bucket, each launch under a time limit that also terminates the
       processes. Does the deadlock depend on the
       overlap of communication with the backward pass, and what does the
       outcome tell you about which workloads the bare benchmark validates?

6. [extended] **Sharded optimizer state.** ● Implement ZeRO stage 1
    [@Rajbhandari.Rasley.Ruwase.ea.2020] by hand and measure what it
    saves. Start from `TRAIN_DDP` with `torch.optim.Adam` in place of SGD.
    1. Assign every parameter tensor to one owner process, so that the
       processes own nearly equal numbers of values, and give each process
       an Adam optimizer over its own tensors only. After the backward pass
       and the owners' steps, send each tensor from its owner to all
       processes with `dist.broadcast`. Check that after a few steps the
       parameters match those of DDP with a full Adam from the same
       initialization, and compare with
       `torch.distributed.optim.ZeroRedundancyOptimizer`.
    1. Read `torch.cuda.memory_allocated` after initialization, before the
       first optimizer step, and after it, with and without sharding at
       $k = 2$ and $k = 4$, and account for the differences with
       the table; then read `torch.cuda.max_memory_allocated`
       after the second step and compare its change under sharding with
       the change in live memory.
    1. Measure the step time with and without sharding. DDP's allreduce
       hands every process the full summed gradient, of which stage 1 uses
       only the part that the process owns. Count the bytes per GPU that the
       allreduce and the broadcasts of your implementation send together,
       and compare them with the reduce-scatter and all-gather of stage 1
       in that section.

    *Adapted from Stanford CS336,
    [Assignment 2 (Systems)](https://github.com/stanford-cs336/assignment2-systems),
    Problems optimizer_state_sharding and
    optimizer_state_sharding_accounting.*

7. [code] **Synchronized batch normalization.** ● The one-step check of
   that section shows that DDP normalizes each GPU's
   shard with the statistics of that shard.
    1. Convert the model in `CHECK_DDP` with
       `nn.SyncBatchNorm.convert_sync_batchnorm` after creating its
       parameters and before wrapping it in `DDP`, and repeat the check.
       Which of the two references does the DDP step now match?
    1. How many collectives per step does the conversion add to
       `resnet18`? Measure `step_time` with the converted model at $k = 2$
       and $k = 4$ under weak scaling, and compare the added time with the
       time that the added bytes take at bandwidth $\beta$. Which
       properties of these collectives, other than their size, set their
       cost?

8. [code] **Tensor versus data parallelism.** ● Consider the MLP block
    $\mathbf{Y} = \phi(\mathbf{X}\mathbf{W}_1)\mathbf{W}_2$ with
    $\mathbf{X} \in \mathbb{R}^{n \times d}$ holding the $n$ tokens of one
    step, $\mathbf{W}_1 \in \mathbb{R}^{d \times 4d}$,
    $\mathbf{W}_2 \in \mathbb{R}^{4d \times d}$, and an elementwise
    activation $\phi$, on $k$ devices. Tensor
    parallelism (the figure) splits $\mathbf{W}_1$ by columns and
    $\mathbf{W}_2$ by rows [@shoeybi2019megatron].
    1. Show that each device can compute, without communication, an
       $n \times d$ partial result from its columns of $\mathbf{W}_1$ and
       its rows of $\mathbf{W}_2$, and that $\mathbf{Y}$ is the sum of the
       $k$ partial results. Which collective forms that sum, and which
       collective does the backward pass need for the gradient with
       respect to $\mathbf{X}$?
    1. Data parallelism on the same $k$ devices allreduces the gradients
       of $\mathbf{W}_1$ and $\mathbf{W}_2$ once per step. Count the values
       that each scheme sends per device and step, and find the number of
       tokens $n$ below which tensor parallelism sends less.
    1. Tensor parallelism is in practice confined to the GPUs of one NVLink
       domain even where it sends less. Explain why, using when each
       scheme's communication happens relative to the computation that
       needs its result.
    1. In a script launched with `d2l.torchrun`, apply `parallelize_module`
       to `nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))`
       on `init_device_mesh("cuda", (k,))`, with `ColwiseParallel()` for
       submodule `"0"` and `RowwiseParallel()` for submodule `"2"`. Count
       the collectives of the forward and the backward pass with
       `CommDebugMode` from `torch.distributed.tensor.debug`, once for an
       input with `requires_grad=True` and once without, and explain the
       difference.

    *Adapted from Stanford CS336,
    [Assignment 2 (Systems)](https://github.com/stanford-cs336/assignment2-systems),
    Problems data_parallel_calcs and tp_calcs.*

[Discussions](https://d2l.discourse.group/t/36792)